In [1]:
!pip install torch torchvision transformers langchain faiss-cpu sentence-transformers pymupdf pillow pdf2image langchain_text_splitters langchain_community

In [2]:
import fitz
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from pdf2image import convert_from_path

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [3]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

device = "cuda" if torch.cuda.is_available() else "cpu"
trocr_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [4]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    # Try normal extraction
    for page in doc:
        text += page.get_text()

    # If little text → use TrOCR
    if len(text.strip()) < 50:
        print("Using TrOCR for handwritten/scanned PDF...")

        images = convert_from_path(pdf_path)
        ocr_text = ""

        for img in images:
            pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(device)

            generated_ids = trocr_model.generate(pixel_values)
            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

            ocr_text += generated_text + "\n"

        return ocr_text

    return text

In [5]:
def chunk_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    return splitter.split_text(text)

In [6]:
# X-ray (Chest + Bone)
xray_model = models.densenet121(pretrained=True)
xray_model.classifier = nn.Linear(xray_model.classifier.in_features, 14)
xray_model.eval()

# MRI + Kidney
mri_model = models.resnet50(pretrained=True)
mri_model.fc = nn.Linear(mri_model.fc.in_features, 4)
mri_model.eval()

# Labels
xray_labels = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass",
    "Nodule","Pneumonia","Pneumothorax","Consolidation","Edema",
    "Emphysema","Fibrosis","Pleural Thickening","Hernia"
]

mri_labels = ["Glioma","Meningioma","Pituitary Tumor","No Tumor"]
kidney_labels = ["Normal", "Stone", "Cyst", "Tumor"]

# Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 102MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 124MB/s]


In [7]:
def analyze_xray(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        probs = torch.sigmoid(xray_model(img))[0]

    findings = [xray_labels[i] for i,p in enumerate(probs) if p > 0.5]

    if not findings:
        return "X-ray: No major abnormality detected"

    return "X-ray Findings: " + ", ".join(findings)

In [8]:
def analyze_bone_xray(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        probs = torch.sigmoid(xray_model(img))[0]

    if torch.max(probs) > 0.6:
        return "Bone X-ray: Possible fracture or abnormality detected"
    else:
        return "Bone X-ray: No obvious fracture detected"

In [9]:
def analyze_mri(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        probs = torch.softmax(mri_model(img), dim=1)[0]

    pred = torch.argmax(probs).item()
    confidence = probs[pred].item()

    return f"MRI Result: {mri_labels[pred]} (confidence {confidence:.2f})"

In [10]:
def analyze_kidney(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        probs = torch.softmax(mri_model(img), dim=1)[0]

    pred = torch.argmax(probs).item()

    return f"Kidney Scan: {kidney_labels[pred]}"

In [11]:
def detect_image_type(filename):
    filename = filename.lower()

    if "bone" in filename:
        return "bone"
    elif "xray" in filename:
        return "xray"
    elif "mri" in filename:
        return "mri"
    elif "kidney" in filename:
        return "kidney"
    else:
        return "unknown"

In [12]:
def analyze_image(path):
    img_type = detect_image_type(path)

    if img_type == "bone":
        return analyze_bone_xray(path)
    elif img_type == "xray":
        return analyze_xray(path)
    elif img_type == "mri":
        return analyze_mri(path)
    elif img_type == "kidney":
        return analyze_kidney(path)
    else:
        return "Unknown image type"

In [13]:
def create_db(texts):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return FAISS.from_texts(texts, embeddings)

In [14]:
def build_knowledge_base(pdf_path=None, image_paths=None, symptoms=None):
    texts = []

    if pdf_path:
        pdf_text = extract_text_from_pdf(pdf_path)
        texts.extend(chunk_text(pdf_text))

    if image_paths:
        for path in image_paths:
            result = analyze_image(path)
            print(result)
            texts.append(result)

    if symptoms:
        texts.append("Symptoms: " + symptoms)

    return create_db(texts)

In [15]:
def process_patient_data(pdf_path=None, image_paths=None, symptoms=None, query=None):

    db = build_knowledge_base(
        pdf_path=pdf_path,
        image_paths=image_paths,
        symptoms=symptoms
    )

    docs = db.similarity_search(query, k=4)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    You are an AI medical assistant.

    Patient Data:
    {context}

    Question:
    {query}

    Instructions:
    - Identify possible disease
    - Suggest next steps
    - Do not give overconfident diagnosis
    - Add disclaimer

    Answer:
    """

    return prompt

In [18]:
prompt = process_patient_data(
    pdf_path="/content/sample_medical_report.pdf",
    image_paths=[
        "/content/xray_OIP.jpg",  # Example: renamed to include 'xray'
        "/content/mri_jpu.jpg",   # Example: renamed to include 'mri'
        "/content/kidney_yyy.webp", # Example: renamed to include 'kidney'
        "/content/bone_yu.webp"   # Example: renamed to include 'bone'
    ],
    symptoms="headache, chest pain, swelling",
    query="What disease is indicated and what should be done?"
)

print(prompt)

X-ray Findings: Atelectasis, Cardiomegaly, Effusion, Nodule, Emphysema
MRI Result: Pituitary Tumor (confidence 0.31)
Kidney Scan: Cyst
Bone X-ray: Possible fracture or abnormality detected


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



    You are an AI medical assistant.

    Patient Data:
    X-ray Findings: Atelectasis, Cardiomegaly, Effusion, Nodule, Emphysema
HOSPITAL NAME: CityCare Multispeciality Hospital
Patient Name: Rahul Sharma
Age/Gender: 45 / Male
Patient ID: CC-2026-0419
Date of Examination: 18 April 2026
Chief Complaints:
- Persistent cough for 10 days
- Mild fever (99–101°F)
- Shortness of breath during exertion
- Fatigue and weakness
Medical History:
- No known history of diabetes
- Mild hypertension (on medication)
- Non-smoker
- No previous major surgeries
Vital Signs:
- Temperature: 100.2°F
- Blood Pressure: 138/88 mmHg
MRI Result: Pituitary Tumor (confidence 0.31)
- Blood Pressure: 138/88 mmHg
- Heart Rate: 92 bpm
- Respiratory Rate: 20 breaths/min
- Oxygen Saturation (SpO2): 94%
Laboratory Investigations:
CBC:
- Hemoglobin: 13.2 g/dL
- WBC Count: 13,500 /µL (Elevated)
- Platelets: Normal
CRP:
- 18 mg/L (Elevated)
Radiology Report (Chest X-ray):
- Patchy opacity in right lower lung
- No pleural 